In [ ]:
%pip install -q --no-cache-dir --force-reinstall --no-deps \
    "numpy==2.1.3" \
    "pandas==2.2.3" \
    "google-auth==2.49.0" \
    "bottleneck==1.5.0" \
    "requests==2.32.4" \
    "python-dateutil==2.9.0.post0" \
    "websockets==15.0.1" \
    "pymongo==4.14.1"

In [ ]:
import pandas as pd
import numpy as np
import pymongo
import google.auth
import requests
import dateutil
import websockets
import bottleneck

packages = {
    "Pandas": pd.__version__,
    "NumPy": np.__version__,
    "PyMongo": pymongo.version,
    "Google Auth": google.auth.__version__,
    "Requests": requests.__version__,
    "Dateutil": dateutil.__version__,
    "Websockets": websockets.__version__,
    "Bottleneck": bottleneck.__version__,
}

for name, version in packages.items():
    print(f"{name}: {version}")

Pandas: 3.1.0
NumPy: 2.1.3
PyMongo: 4.14.1
Google Auth: 2.62.0
Requests: 2.32.4
Dateutil: 2.9.0.post0
Websockets: 15.0.1
Bottleneck: 1.4.2


In [ ]:
import os
from pathlib import Path

# Dataset configuration
DATASET_PATH = "off.csv.gz"

# MongoDB configuration
DB_NAME = "food_database"
COLLECTION_NAME = "products"

# Output files
RAW_PARQUET = "india_raw.parquet"
CLEAN_PARQUET = "india_clean.parquet"

# Processing configuration
CHUNK_SIZE = 100_000
BATCH_SIZE = 1000

# Gemini configuration
GEMINI_MODEL = "gemini-2.5-flash"

# PDF processing
PROCESS_PDFS = True
PDF_DIRECTORY = Path("raw")
PDF_DIRECTORY.mkdir(parents=True, exist_ok=True)

print("Configuration loaded successfully")
print("Dataset:", DATASET_PATH)
print("Database:", DB_NAME)
print("Collection:", COLLECTION_NAME)
print("Chunk size:", CHUNK_SIZE)
print("Batch size:", BATCH_SIZE)
print("Gemini model:", GEMINI_MODEL)
print("PDF directory:", PDF_DIRECTORY)

Configuration loaded successfully
Dataset: off.csv.gz
Database: food_database
Collection: products
Chunk size: 100000
Batch size: 1000
Gemini model: gemini-2.5-flash
PDF directory: raw


In [ ]:
from google.colab import files
from pathlib import Path

uploaded = files.upload()

if not uploaded:
    raise ValueError("No file uploaded")

DATASET_PATH = next(iter(uploaded.keys()))

if not DATASET_PATH.lower().endswith((".csv", ".csv.gz")):
    raise ValueError("Please upload a CSV or CSV.GZ file")

print("Dataset:", DATASET_PATH)

size_mb = Path(DATASET_PATH).stat().st_size / (1024 * 1024)

print(f"File size: {size_mb:.2f} MB")
print("Upload successful!")

Saving packaged_foods_india.csv to packaged_foods_india (1).csv
Dataset: packaged_foods_india (1).csv
File size: 0.32 MB
Upload successful!


In [ ]:
import pandas as pd

# Read actual CSV headers
header = pd.read_csv(
    DATASET_PATH,
    sep=",",
    nrows=0
).columns.tolist()

# Expected columns
WANTED_COLUMNS = [
    "Item name",
    "Brand_Name",
    "Category",
    "Sub_Category",
    "Ingredients",
    "Serving_Size_g",
    "Calories_kcal",
    "Carbohydrates_g",
    "Proteins_g",
    "Total_Fat_g",
    "Saturated_Fat_g",
    "Trans_Fat_g",
    "Sugar_g",
    "Sodium_mg",
    "Dietary_Fiber_g",
    "Cholesterol_mg",
    "Calcium_mg",
    "Vitamin",
    "Iron_mg",
    "Price_INR",
]

USE_COLUMNS = [
    col for col in WANTED_COLUMNS
    if col in header
]

missing = set(WANTED_COLUMNS) - set(USE_COLUMNS)

print("Total CSV columns:", len(header))
print("Available columns:", len(USE_COLUMNS))
print("Missing columns:", missing)

# Mandatory columns
required = {
    "Item name",
    "Category",
    "Calories_kcal",
}

missing_required = required - set(USE_COLUMNS)

if missing_required:
    raise ValueError(
        f"Missing mandatory columns: {missing_required}"
    )

print("\nAll mandatory columns found!")

# Display sample
sample = pd.read_csv(
    DATASET_PATH,
    usecols=USE_COLUMNS,
    nrows=5
)

display(sample)

Total CSV columns: 25
Available columns: 20
Missing columns: set()

All mandatory columns found!


,Item name,Brand_Name,Category,Sub_Category,Ingredients,Serving_Size_g,Calories_kcal,Carbohydrates_g,Proteins_g,Total_Fat_g,Saturated_Fat_g,Trans_Fat_g,Sugar_g,Sodium_mg,Dietary_Fiber_g,Cholesterol_mg,Calcium_mg,Vitamin,Iron_mg,Price_INR
0,HIDE & SEEK®,PARLE,BISCUIT,COOKIES,"REFINED WHEAT FLOUR (MAIDA), CHOCOLATE (23%) ...",100,479,73.4,5.9,18.0,9.4,0.00,32.5,114.0,0.0,0.0,0,0,0.0,30
1,Britannia Nutrichoice Digestive High-Fibre Bis...,BRITANNIA,BISCUIT,DIGESTIVE BISCUIT,"WHEAT FLOUR (ATTA) (65.5%), REFINED PALM OIL, ...",100,483,68.0,9.3,20.8,10.3,0.00,12.7,362.0,6.7,0.1,0,0,0.0,20
2,"Sunfeast Dark Fantasy Choco Fills, Original Fi...",SUNFEAST,BISCUIT,CHOCO FILLS,"SUGAR, REFINED PALMOLEIN, REFINED PALM OIL, C...",100,507,61.1,7.3,27.3,12.7,0.10,35.7,181.5,0.0,0.3,0,0,0.0,170
3,Sunfeast Baked Creations Rich Choco Chip Cookies,SUNFEAST,BISCUIT,COOKIES,"MAIDA (REFINED WHEAT FLOUR), DARK CHOCOLATE CH...",20,105,12.6,1.2,5.5,3.2,0.03,6.8,35.1,0.0,1.0,0,0,0.0,25
4,"sunfeast Dark Fantasy Bourbon, Classic Biscuit...",SUNFEAST,BISCUIT,CREAME,"Cocoa Solids (2.3%), Emulsifiers lecithin (fro...",108,445,77.4,9.1,11.0,5.6,0.07,21.3,292.4,0.0,0.1,0,"Vitamin B1 (mg) 0.3, Vitamin B2 (mg0 0.3, Vita...",2.9,35


In [ ]:
import pandas as pd
import gc

parts = []

reader = pd.read_csv(
    DATASET_PATH,
    sep=",",
    usecols=USE_COLUMNS,
    dtype=str,
    chunksize=CHUNK_SIZE,
    on_bad_lines="skip",
    encoding_errors="replace",
    low_memory=False
)

total_rows = 0

for chunk in reader:
    total_rows += len(chunk)

    chunk["Item name"] = (
        chunk["Item name"]
        .fillna("")
        .str.strip()
    )

    chunk = chunk[
        chunk["Item name"] != ""
    ].copy()

    if not chunk.empty:
        parts.append(chunk)

    del chunk

if not parts:
    raise ValueError("No valid products found")

india = pd.concat(
    parts,
    ignore_index=True
)

del parts
gc.collect()

# Save as CSV instead of Parquet
india.to_csv(
    "india_raw.csv",
    index=False
)

print("Total rows scanned:", total_rows)
print("Valid products:", len(india))
print("Total columns:", len(india.columns))
print("Saved: india_raw.csv")

display(india.head(10))

Total rows scanned: 852
Valid products: 852
Total columns: 20
Saved: india_raw.csv


,Item name,Brand_Name,Category,Sub_Category,Ingredients,Serving_Size_g,Calories_kcal,Carbohydrates_g,Proteins_g,Total_Fat_g,Saturated_Fat_g,Trans_Fat_g,Sugar_g,Sodium_mg,Dietary_Fiber_g,Cholesterol_mg,Calcium_mg,Vitamin,Iron_mg,Price_INR
0,HIDE & SEEK®,PARLE,BISCUIT,COOKIES,"REFINED WHEAT FLOUR (MAIDA), CHOCOLATE (23%) ...",100,479,73.4,5.9,18,9.4,0,32.5,114,0,0,0,0,0,30
1,Britannia Nutrichoice Digestive High-Fibre Bis...,BRITANNIA,BISCUIT,DIGESTIVE BISCUIT,"WHEAT FLOUR (ATTA) (65.5%), REFINED PALM OIL, ...",100,483,68,9.3,20.8,10.3,0,12.7,362,6.7,0.1,0,0,0,20
2,"Sunfeast Dark Fantasy Choco Fills, Original Fi...",SUNFEAST,BISCUIT,CHOCO FILLS,"SUGAR, REFINED PALMOLEIN, REFINED PALM OIL, C...",100,507,61.1,7.3,27.3,12.7,0.1,35.7,181.5,0,0.3,0,0,0,170
3,Sunfeast Baked Creations Rich Choco Chip Cookies,SUNFEAST,BISCUIT,COOKIES,"MAIDA (REFINED WHEAT FLOUR), DARK CHOCOLATE CH...",20,105,12.6,1.2,5.5,3.2,0.03,6.8,35.1,0,1,0,0,0,25
4,"sunfeast Dark Fantasy Bourbon, Classic Biscuit...",SUNFEAST,BISCUIT,CREAME,"Cocoa Solids (2.3%), Emulsifiers lecithin (fro...",108,445,77.4,9.1,11,5.6,0.07,21.3,292.4,0,0.1,0,"Vitamin B1 (mg) 0.3, Vitamin B2 (mg0 0.3, Vita...",2.9,35
5,Britannia Good Day Cashew Cookies Family Pack,BRITANNIA,BISCUIT,COOKIES,Added Flavours (Nature Identical and Artificia...,100,512,67,7,24,11,0,22,0,0,2.5,0,0,0,25
6,The Baker's Dozen Zero Maida Coconut and Oats ...,THE BAKER'S DOZEN,BISCUIT,COOKIES,Artificial Flavouring Substance (VANILLA CAKE)...,100,538,56,10,30,14,0,25,160,8,0,0,0,NaN,NaN
7,Britannia The Original Bourbon,BRITANNIA,BISCUIT,CREAME,"Refined Wheat Flour, Sugar, Refined Palm Oil, ...",100,493.5,72.7,5,20.3,10.1,0,38.3,119,0,0,0,0,0,25
8,Nutrisum Oats Cookies,NUTRISUM,BISCUIT,COOKIES,"Oats Flour (25%), Wheat Flour (Atta) (25%), Hy...",100,516.9,15.4,8.2,27.3,11.8,0,5,203.7,12,0,0,0,0,95
9,Britannia Good Day Pista Badam,BRITANNIA,BISCUIT,COOKIES,"Refined Wheat Flour (Maida), Sugar, Edible Veg...",100,512,67,7,24,11,0,22,0,0,2,0,0,0,25


In [ ]:
import pandas as pd
import numpy as np

# Nutritional columns in your CSV
NUMERIC_COLUMNS = [
    "Serving_Size_g",
    "Calories_kcal",
    "Carbohydrates_g",
    "Proteins_g",
    "Total_Fat_g",
    "Saturated_Fat_g",
    "Trans_Fat_g",
    "Sugar_g",
    "Sodium_mg",
    "Dietary_Fiber_g",
    "Cholesterol_mg",
    "Calcium_mg",
    "Iron_mg",
    "Price_INR",
]

# Convert values into numeric format
for col in NUMERIC_COLUMNS:
    if col in india.columns:
        india[col] = pd.to_numeric(
            india[col]
            .astype(str)
            .str.replace(",", "", regex=False)
            .str.strip()
            .replace({
                "": None,
                "nan": None,
                "None": None,
                "N/A": None,
                "NA": None,
                "-": None,
            }),
            errors="coerce"
        )

# Display missing values
print("Numeric conversion complete!\n")

print("Missing values per column:")
display(india[NUMERIC_COLUMNS].isna().sum().to_frame("Missing"))

print("\nSample converted nutritional data:")
display(india[NUMERIC_COLUMNS].head(10))

Numeric conversion complete!

Missing values per column:


,Missing
Serving_Size_g,0
Calories_kcal,1
Carbohydrates_g,1
Proteins_g,3
Total_Fat_g,7
Saturated_Fat_g,10
Trans_Fat_g,20
Sugar_g,8
Sodium_mg,6
Dietary_Fiber_g,10



Sample converted nutritional data:


,Serving_Size_g,Calories_kcal,Carbohydrates_g,Proteins_g,Total_Fat_g,Saturated_Fat_g,Trans_Fat_g,Sugar_g,Sodium_mg,Dietary_Fiber_g,Cholesterol_mg,Calcium_mg,Iron_mg,Price_INR
0,100.0,479.0,73.4,5.9,18.0,9.4,0.00,32.5,114.0,0.0,0.0,0.0,0.0,30.0
1,100.0,483.0,68.0,9.3,20.8,10.3,0.00,12.7,362.0,6.7,0.1,0.0,0.0,20.0
2,100.0,507.0,61.1,7.3,27.3,12.7,0.10,35.7,181.5,0.0,0.3,0.0,0.0,170.0
3,20.0,105.0,12.6,1.2,5.5,3.2,0.03,6.8,35.1,0.0,1.0,0.0,0.0,25.0
4,108.0,445.0,77.4,9.1,11.0,5.6,0.07,21.3,292.4,0.0,0.1,0.0,2.9,35.0
5,100.0,512.0,67.0,7.0,24.0,11.0,0.00,22.0,0.0,0.0,2.5,0.0,0.0,25.0
6,100.0,538.0,56.0,10.0,30.0,14.0,0.00,25.0,160.0,8.0,0.0,0.0,NaN,NaN
7,100.0,493.5,72.7,5.0,20.3,10.1,0.00,38.3,119.0,0.0,0.0,0.0,0.0,25.0
8,100.0,516.9,15.4,8.2,27.3,11.8,0.00,5.0,203.7,12.0,0.0,0.0,0.0,95.0
9,100.0,512.0,67.0,7.0,24.0,11.0,0.00,22.0,0.0,0.0,2.0,0.0,0.0,25.0


In [ ]:
# Columns to normalize
NUTRIENT_COLUMNS = [
    "Calories_kcal",
    "Carbohydrates_g",
    "Proteins_g",
    "Total_Fat_g",
    "Saturated_Fat_g",
    "Trans_Fat_g",
    "Sugar_g",
    "Sodium_mg",
    "Dietary_Fiber_g",
    "Cholesterol_mg",
    "Calcium_mg",
    "Iron_mg",
]

# Preserve original serving-based values
for col in NUTRIENT_COLUMNS:
    india[f"original_{col}"] = india[col]

# Normalize to per 100g
valid_serving = (
    india["Serving_Size_g"].notna()
    & india["Serving_Size_g"].gt(0)
)

for col in NUTRIENT_COLUMNS:
    india[f"{col}_per100g"] = np.where(
        valid_serving,
        india[col] * 100 / india["Serving_Size_g"],
        np.nan
    )

print("Nutrition normalization complete!")

print("\nServing sizes:")
display(india["Serving_Size_g"].describe())

print("\nNormalized nutrition sample:")
display(
    india[
        [
            "Item name",
            "Serving_Size_g",
            "Calories_kcal_per100g",
            "Carbohydrates_g_per100g",
            "Proteins_g_per100g",
            "Total_Fat_g_per100g",
            "Sugar_g_per100g",
            "Sodium_mg_per100g",
        ]
    ].head(10)
)

Nutrition normalization complete!

Serving sizes:


count    852.000000
mean      96.601373
std       27.031715
min        2.000000
25%      100.000000
50%      100.000000
75%      100.000000
max      585.770000
Name: Serving_Size_g, dtype: float64


Normalized nutrition sample:


,Item name,Serving_Size_g,Calories_kcal_per100g,Carbohydrates_g_per100g,Proteins_g_per100g,Total_Fat_g_per100g,Sugar_g_per100g,Sodium_mg_per100g
0,HIDE & SEEK®,100.0,479.000000,73.400000,5.900000,18.000000,32.500000,114.000000
1,Britannia Nutrichoice Digestive High-Fibre Bis...,100.0,483.000000,68.000000,9.300000,20.800000,12.700000,362.000000
2,"Sunfeast Dark Fantasy Choco Fills, Original Fi...",100.0,507.000000,61.100000,7.300000,27.300000,35.700000,181.500000
3,Sunfeast Baked Creations Rich Choco Chip Cookies,20.0,525.000000,63.000000,6.000000,27.500000,34.000000,175.500000
4,"sunfeast Dark Fantasy Bourbon, Classic Biscuit...",108.0,412.037037,71.666667,8.425926,10.185185,19.722222,270.740741
5,Britannia Good Day Cashew Cookies Family Pack,100.0,512.000000,67.000000,7.000000,24.000000,22.000000,0.000000
6,The Baker's Dozen Zero Maida Coconut and Oats ...,100.0,538.000000,56.000000,10.000000,30.000000,25.000000,160.000000
7,Britannia The Original Bourbon,100.0,493.500000,72.700000,5.000000,20.300000,38.300000,119.000000
8,Nutrisum Oats Cookies,100.0,516.900000,15.400000,8.200000,27.300000,5.000000,203.700000
9,Britannia Good Day Pista Badam,100.0,512.000000,67.000000,7.000000,24.000000,22.000000,0.000000


In [ ]:
# Nutritional columns after per-100g normalization
PER100G_COLUMNS = [
    "Calories_kcal_per100g",
    "Carbohydrates_g_per100g",
    "Proteins_g_per100g",
    "Total_Fat_g_per100g",
    "Saturated_Fat_g_per100g",
    "Trans_Fat_g_per100g",
    "Sugar_g_per100g",
    "Sodium_mg_per100g",
    "Dietary_Fiber_g_per100g",
    "Cholesterol_mg_per100g",
    "Calcium_mg_per100g",
    "Iron_mg_per100g",
]

# Check impossible negative values
negative_mask = (
    india[PER100G_COLUMNS].lt(0).any(axis=1)
)

# Broad plausibility limits
limits = {
    "Calories_kcal_per100g": 900,
    "Carbohydrates_g_per100g": 100,
    "Proteins_g_per100g": 100,
    "Total_Fat_g_per100g": 100,
    "Saturated_Fat_g_per100g": 100,
    "Trans_Fat_g_per100g": 100,
    "Sugar_g_per100g": 100,
    "Sodium_mg_per100g": 40000,
    "Dietary_Fiber_g_per100g": 100,
    "Cholesterol_mg_per100g": 5000,
    "Calcium_mg_per100g": 10000,
    "Iron_mg_per100g": 1000,
}

# Identify values exceeding limits
suspicious_mask = pd.Series(
    False,
    index=india.index
)

for col, limit in limits.items():
    suspicious_mask |= india[col].gt(limit)

# Flag records instead of immediately deleting them
india["dataQuality"] = np.select(
    [
        negative_mask,
        suspicious_mask
    ],
    [
        "invalid_negative_value",
        "needs_review"
    ],
    default="valid"
)

# Separate records for inspection
rejected = india[negative_mask].copy()
review = india[suspicious_mask & ~negative_mask].copy()
clean = india[~negative_mask].copy()

print("Total products:", len(india))
print("Valid products:", len(clean))
print("Negative-value records:", len(rejected))
print("Suspicious records:", len(review))

print("\nSuspicious products:")
display(
    review[
        ["Item name", "Category"] + PER100G_COLUMNS
    ].head(10)
)

Total products: 852
Valid products: 852
Negative-value records: 0
Suspicious records: 12

Suspicious products:


,Item name,Category,Calories_kcal_per100g,Carbohydrates_g_per100g,Proteins_g_per100g,Total_Fat_g_per100g,Saturated_Fat_g_per100g,Trans_Fat_g_per100g,Sugar_g_per100g,Sodium_mg_per100g,Dietary_Fiber_g_per100g,Cholesterol_mg_per100g,Calcium_mg_per100g,Iron_mg_per100g
69,Britannia Good Day Chunkies Coconut Cookies,BISCUIT,575.000000,532.800000,6.200000,37.20,24.700000,0.0,28.000000,143.000000,0.000000,13.10,0.0,0.00
317,Mr Makhana Super Snack - Pudina Party,SNACKS,514.600000,58.280000,765.000000,27.86,6.900000,0.0,0.500000,190.000000,2.450000,0.00,6.0,4.53
423,Darkins Indian Artisanal 63% Dark Orange Choco...,CHOCOLATE,1342.000000,138.000000,17.000000,82.00,49.000000,0.0,89.000000,0.000000,0.000000,0.00,0.0,0.00
426,Darkins Indian Artisanal 70% Dark Cranberry & ...,CHOCOLATE,309.000000,141.000000,18.000000,79.00,49.000000,0.0,62.000000,0.000000,0.000000,0.00,0.0,0.00
427,Darkins Indian Artisanal 70% Dark Roasted Almo...,CHOCOLATE,1440.000000,118.000000,26.000000,99.00,47.000000,0.0,63.000000,0.000000,37.000000,0.00,0.0,0.00
679,Gladful Choco Pancake Sprouted Millets and Mas...,INSTANT FOOD,1023.333333,206.333333,33.333333,7.00,0.000000,0.0,63.000000,1200.000000,15.000000,0.00,0.0,0.00
704,Betty Crocker Triple Chocolate Brownie Instant...,INSTANT FOOD,1634.000000,81.600000,3.400000,4.60,2.200000,0.1,54.300000,228.000000,0.000000,0.00,0.0,0.00
747,Wingreens Farms Tandoori Mayo,SAUCE,281.000000,14.400000,1.000000,24.40,2.200000,0.0,0.500000,8.100000,653.700000,0.31,0.0,0.00
765,ADF Soul Sriracha Dip,SPREAD,513.333333,115.333333,6.000000,4.00,1.333333,0.0,100.666667,4026.666667,5.333333,0.00,0.0,0.00
766,ADF Soul Twisty Tamarind Dip,SPREAD,1513.333333,338.666667,8.000000,20.00,2.666667,0.0,280.666667,3720.000000,24.000000,0.00,0.0,0.00


In [ ]:
import re

PALM_PATTERN = re.compile(
    r"\bpalm\s*oil\b|"
    r"\bpalmolein\b|"
    r"\bpalm\s*olein\b|"
    r"\bpalm\s*kernel\b|"
    r"\bpalm\s*fat\b",
    re.IGNORECASE
)

POSSIBLE_PATTERN = re.compile(
    r"\bvegetable\s*(?:fat|oil)\b|"
    r"\bedible\s*vegetable\s*(?:fat|oil)\b|"
    r"\bvegetable\s*shortening\b",
    re.IGNORECASE
)

def detect_palm_oil(ingredients):

    if not isinstance(ingredients, str) or not ingredients.strip():
        return {
            "palmOil": None,
            "palmOilStatus": "unknown"
        }

    if PALM_PATTERN.search(ingredients):
        return {
            "palmOil": True,
            "palmOilStatus": "confirmed"
        }

    if POSSIBLE_PATTERN.search(ingredients):
        return {
            "palmOil": None,
            "palmOilStatus": "possible"
        }

    return {
        "palmOil": False,
        "palmOilStatus": "not_detected"
    }


palm_results = clean["Ingredients"].apply(detect_palm_oil)

clean["palmOil"] = palm_results.apply(
    lambda x: x["palmOil"]
)

clean["palmOilStatus"] = palm_results.apply(
    lambda x: x["palmOilStatus"]
)

print("Palm oil detection completed!")

display(
    clean[
        ["Item name", "Ingredients", "palmOil", "palmOilStatus"]
    ].head(15)
)

print("\nPalm oil statistics:")
display(clean["palmOilStatus"].value_counts())

Palm oil detection completed!


,Item name,Ingredients,palmOil,palmOilStatus
0,HIDE & SEEK®,"REFINED WHEAT FLOUR (MAIDA), CHOCOLATE (23%) ...",True,confirmed
1,Britannia Nutrichoice Digestive High-Fibre Bis...,"WHEAT FLOUR (ATTA) (65.5%), REFINED PALM OIL, ...",True,confirmed
2,"Sunfeast Dark Fantasy Choco Fills, Original Fi...","SUGAR, REFINED PALMOLEIN, REFINED PALM OIL, C...",True,confirmed
3,Sunfeast Baked Creations Rich Choco Chip Cookies,"MAIDA (REFINED WHEAT FLOUR), DARK CHOCOLATE CH...",True,confirmed
4,"sunfeast Dark Fantasy Bourbon, Classic Biscuit...","Cocoa Solids (2.3%), Emulsifiers lecithin (fro...",True,confirmed
5,Britannia Good Day Cashew Cookies Family Pack,Added Flavours (Nature Identical and Artificia...,None,possible
6,The Baker's Dozen Zero Maida Coconut and Oats ...,Artificial Flavouring Substance (VANILLA CAKE)...,False,not_detected
7,Britannia The Original Bourbon,"Refined Wheat Flour, Sugar, Refined Palm Oil, ...",True,confirmed
8,Nutrisum Oats Cookies,"Oats Flour (25%), Wheat Flour (Atta) (25%), Hy...",None,possible
9,Britannia Good Day Pista Badam,"Refined Wheat Flour (Maida), Sugar, Edible Veg...",None,possible



Palm oil statistics:


palmOilStatus
not_detected    502
confirmed       227
possible        123
Name: count, dtype: int64

In [ ]:
import hashlib

def generate_product_id(row):
    raw = "|".join([
        str(row.get("Item name", "")).strip().lower(),
        str(row.get("Brand_Name", "")).strip().lower(),
        str(row.get("Category", "")).strip().lower(),
        str(row.get("Sub_Category", "")).strip().lower()
    ])

    return hashlib.sha256(
        raw.encode("utf-8")
    ).hexdigest()[:24]


clean["productId"] = clean.apply(
    generate_product_id,
    axis=1
)

# Check duplicates
duplicate_mask = clean.duplicated(
    subset=["productId"],
    keep="first"
)

duplicates = clean[duplicate_mask].copy()

# Keep one record per product ID
clean = clean.drop_duplicates(
    subset=["productId"],
    keep="first"
).copy()

print("Products before deduplication:", len(duplicate_mask) + len(clean) - len(duplicates))
print("Duplicates removed:", len(duplicates))
print("Unique products:", len(clean))

display(
    clean[
        ["productId", "Item name", "Brand_Name", "Category"]
    ].head(10)
)

Products before deduplication: 1686
Duplicates removed: 9
Unique products: 843


,productId,Item name,Brand_Name,Category
0,e013f3bd19cb30a368077757,HIDE & SEEK®,PARLE,BISCUIT
1,f9e612e897a36ee5d602fb26,Britannia Nutrichoice Digestive High-Fibre Bis...,BRITANNIA,BISCUIT
2,80e210c69dd05557c833deb9,"Sunfeast Dark Fantasy Choco Fills, Original Fi...",SUNFEAST,BISCUIT
3,dbd66914771daa8fd1cc7579,Sunfeast Baked Creations Rich Choco Chip Cookies,SUNFEAST,BISCUIT
4,75b6c07a865128032277cded,"sunfeast Dark Fantasy Bourbon, Classic Biscuit...",SUNFEAST,BISCUIT
5,5f50b0bad439c1d7d645fa80,Britannia Good Day Cashew Cookies Family Pack,BRITANNIA,BISCUIT
6,01775f87dc8bf0fc1bcf234b,The Baker's Dozen Zero Maida Coconut and Oats ...,THE BAKER'S DOZEN,BISCUIT
7,4924fd1ffcaf816c477e2579,Britannia The Original Bourbon,BRITANNIA,BISCUIT
8,bc51b23572df14dd4f3d487b,Nutrisum Oats Cookies,NUTRISUM,BISCUIT
9,7ae0e6f2d95d26e2787ed7e2,Britannia Good Day Pista Badam,BRITANNIA,BISCUIT


In [ ]:
# Standardized nutrition column mapping
NUTRITION_MAPPING = {
    "caloriesKcal": "Calories_kcal_per100g",
    "carbs": "Carbohydrates_g_per100g",
    "protein": "Proteins_g_per100g",
    "totalFat": "Total_Fat_g_per100g",
    "satFat": "Saturated_Fat_g_per100g",
    "transFat": "Trans_Fat_g_per100g",
    "sugars": "Sugar_g_per100g",
    "sodiumMg": "Sodium_mg_per100g",
    "fibre": "Dietary_Fiber_g_per100g",
    "cholesterolMg": "Cholesterol_mg_per100g",
    "calciumMg": "Calcium_mg_per100g",
    "ironMg": "Iron_mg_per100g",
}

def safe_number(value):
    if pd.isna(value):
        return None

    value = float(value)

    if not np.isfinite(value):
        return None

    return round(value, 2)


# Create standardized nutrition dictionary
clean["nutrition"] = clean.apply(
    lambda row: {
        key: safe_number(row[column])
        for key, column in NUTRITION_MAPPING.items()
    },
    axis=1
)

# Create standardized product columns
clean["name"] = clean["Item name"].str.strip()

clean["brand"] = clean["Brand_Name"].where(
    clean["Brand_Name"].notna(),
    None
)

clean["category"] = clean["Category"]

clean["subCategory"] = clean["Sub_Category"]

clean["ingredientsText"] = clean["Ingredients"]

clean["servingSizeG"] = clean["Serving_Size_g"].apply(
    safe_number
)

clean["priceINR"] = clean["Price_INR"].apply(
    safe_number
)

# Preview final structure
display(
    clean[
        [
            "productId",
            "name",
            "brand",
            "category",
            "subCategory",
            "nutrition",
            "palmOil",
            "palmOilStatus",
            "servingSizeG",
            "priceINR",
        ]
    ].head(5)
)

print("Final nutrition structures created:", len(clean))

,productId,name,brand,category,subCategory,nutrition,palmOil,palmOilStatus,servingSizeG,priceINR
0,e013f3bd19cb30a368077757,HIDE & SEEK®,PARLE,BISCUIT,COOKIES,"{'caloriesKcal': 479.0, 'carbs': 73.4, 'protei...",True,confirmed,100.0,30.0
1,f9e612e897a36ee5d602fb26,Britannia Nutrichoice Digestive High-Fibre Bis...,BRITANNIA,BISCUIT,DIGESTIVE BISCUIT,"{'caloriesKcal': 483.0, 'carbs': 68.0, 'protei...",True,confirmed,100.0,20.0
2,80e210c69dd05557c833deb9,"Sunfeast Dark Fantasy Choco Fills, Original Fi...",SUNFEAST,BISCUIT,CHOCO FILLS,"{'caloriesKcal': 507.0, 'carbs': 61.1, 'protei...",True,confirmed,100.0,170.0
3,dbd66914771daa8fd1cc7579,Sunfeast Baked Creations Rich Choco Chip Cookies,SUNFEAST,BISCUIT,COOKIES,"{'caloriesKcal': 525.0, 'carbs': 63.0, 'protei...",True,confirmed,20.0,25.0
4,75b6c07a865128032277cded,"sunfeast Dark Fantasy Bourbon, Classic Biscuit...",SUNFEAST,BISCUIT,CREAME,"{'caloriesKcal': 412.04, 'carbs': 71.67, 'prot...",True,confirmed,108.0,35.0


Final nutrition structures created: 843


In [ ]:
# Check product IDs
valid_ids = clean["productId"].notna()

# Check product names
valid_names = (
    clean["name"].notna()
    & clean["name"].str.strip().ne("")
)

# Check palm oil values
valid_palm = clean["palmOil"].apply(
    lambda x: isinstance(x, (bool, np.bool_))
    or pd.isna(x)
)

# Check nutrition dictionaries
valid_nutrition = clean["nutrition"].apply(
    lambda x: isinstance(x, dict)
)

# Combined validation
valid_records = (
    valid_ids
    & valid_names
    & valid_palm
    & valid_nutrition
)

invalid_records = clean[~valid_records].copy()

final_df = clean[valid_records].copy()

# Export
final_df.to_csv(
    CLEAN_PARQUET.replace(".parquet", "_final.csv"),
    index=False
)

print("Total records:", len(clean))
print("Valid records:", len(final_df))
print("Invalid records:", len(invalid_records))
print("Unique product IDs:", final_df["productId"].nunique())

print("\nPalm oil summary:")
display(final_df["palmOilStatus"].value_counts())

print("\nFinal dataset preview:")
display(
    final_df[
        [
            "productId",
            "name",
            "brand",
            "category",
            "nutrition",
            "palmOil",
            "palmOilStatus",
        ]
    ].head(10)
)

Total records: 843
Valid records: 843
Invalid records: 0
Unique product IDs: 843

Palm oil summary:


palmOilStatus
not_detected    498
confirmed       224
possible        121
Name: count, dtype: int64


Final dataset preview:


,productId,name,brand,category,nutrition,palmOil,palmOilStatus
0,e013f3bd19cb30a368077757,HIDE & SEEK®,PARLE,BISCUIT,"{'caloriesKcal': 479.0, 'carbs': 73.4, 'protei...",True,confirmed
1,f9e612e897a36ee5d602fb26,Britannia Nutrichoice Digestive High-Fibre Bis...,BRITANNIA,BISCUIT,"{'caloriesKcal': 483.0, 'carbs': 68.0, 'protei...",True,confirmed
2,80e210c69dd05557c833deb9,"Sunfeast Dark Fantasy Choco Fills, Original Fi...",SUNFEAST,BISCUIT,"{'caloriesKcal': 507.0, 'carbs': 61.1, 'protei...",True,confirmed
3,dbd66914771daa8fd1cc7579,Sunfeast Baked Creations Rich Choco Chip Cookies,SUNFEAST,BISCUIT,"{'caloriesKcal': 525.0, 'carbs': 63.0, 'protei...",True,confirmed
4,75b6c07a865128032277cded,"sunfeast Dark Fantasy Bourbon, Classic Biscuit...",SUNFEAST,BISCUIT,"{'caloriesKcal': 412.04, 'carbs': 71.67, 'prot...",True,confirmed
5,5f50b0bad439c1d7d645fa80,Britannia Good Day Cashew Cookies Family Pack,BRITANNIA,BISCUIT,"{'caloriesKcal': 512.0, 'carbs': 67.0, 'protei...",None,possible
6,01775f87dc8bf0fc1bcf234b,The Baker's Dozen Zero Maida Coconut and Oats ...,THE BAKER'S DOZEN,BISCUIT,"{'caloriesKcal': 538.0, 'carbs': 56.0, 'protei...",False,not_detected
7,4924fd1ffcaf816c477e2579,Britannia The Original Bourbon,BRITANNIA,BISCUIT,"{'caloriesKcal': 493.5, 'carbs': 72.7, 'protei...",True,confirmed
8,bc51b23572df14dd4f3d487b,Nutrisum Oats Cookies,NUTRISUM,BISCUIT,"{'caloriesKcal': 516.9, 'carbs': 15.4, 'protei...",None,possible
9,7ae0e6f2d95d26e2787ed7e2,Britannia Good Day Pista Badam,BRITANNIA,BISCUIT,"{'caloriesKcal': 512.0, 'carbs': 67.0, 'protei...",None,possible


In [ ]:
from google.colab import userdata
from google import genai
from google.genai import types
from pydantic import BaseModel
import json
import time

# Retrieve API key from Colab Secrets
GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")

if not GEMINI_API_KEY:
    raise ValueError("GEMINI_API_KEY not found in Colab Secrets")

# Initialize Gemini
GEMINI_MODEL = "gemini-2.5-flash"

gemini_client = genai.Client(
    api_key=GEMINI_API_KEY
)

print("Gemini initialized successfully!")
print("Model:", GEMINI_MODEL)

Gemini initialized successfully!
Model: gemini-2.5-flash


In [ ]:
# Cell 18: Merge Gemini results safely

import pandas as pd

# Ensure Gemini results exist
if "gemini_results" not in globals():
    gemini_results = {}

# Start with cleaned dataset
enriched_df = final_df.copy()

# Initialize enrichment columns
enrichment_columns = {
    "geminiPalmOil": None,
    "geminiPalmOilStatus": "not_analyzed",
    "vegetarian": None,
    "allergenInfo": None,
    "ingredientSummary": None,
    "geminiConfidence": None,
}

for column, default in enrichment_columns.items():
    enriched_df[column] = [default] * len(enriched_df)

# Merge available Gemini results
for idx, row in enriched_df.iterrows():

    result = gemini_results.get(row["productId"])

    if not result:
        continue

    enriched_df.at[idx, "geminiPalmOil"] = result.get("palmOil")
    enriched_df.at[idx, "geminiPalmOilStatus"] = result.get("palmOilStatus")
    enriched_df.at[idx, "vegetarian"] = result.get("vegetarian")
    enriched_df.at[idx, "allergenInfo"] = result.get("allergenInfo")
    enriched_df.at[idx, "ingredientSummary"] = result.get("ingredientSummary")
    enriched_df.at[idx, "geminiConfidence"] = result.get("confidence")

print("Merge completed!")
print("Total products:", len(enriched_df))
print("Gemini analyzed:", len(gemini_results))

display(
    enriched_df[
        [
            "name",
            "brand",
            "palmOil",
            "palmOilStatus",
            "geminiPalmOil",
            "vegetarian",
            "allergenInfo"
        ]
    ].head(10)
)

Merge completed!
Total products: 843
Gemini analyzed: 0


,name,brand,palmOil,palmOilStatus,geminiPalmOil,vegetarian,allergenInfo
0,HIDE & SEEK®,PARLE,True,confirmed,None,None,None
1,Britannia Nutrichoice Digestive High-Fibre Bis...,BRITANNIA,True,confirmed,None,None,None
2,"Sunfeast Dark Fantasy Choco Fills, Original Fi...",SUNFEAST,True,confirmed,None,None,None
3,Sunfeast Baked Creations Rich Choco Chip Cookies,SUNFEAST,True,confirmed,None,None,None
4,"sunfeast Dark Fantasy Bourbon, Classic Biscuit...",SUNFEAST,True,confirmed,None,None,None
5,Britannia Good Day Cashew Cookies Family Pack,BRITANNIA,None,possible,None,None,None
6,The Baker's Dozen Zero Maida Coconut and Oats ...,THE BAKER'S DOZEN,False,not_detected,None,None,None
7,Britannia The Original Bourbon,BRITANNIA,True,confirmed,None,None,None
8,Nutrisum Oats Cookies,NUTRISUM,None,possible,None,None,None
9,Britannia Good Day Pista Badam,BRITANNIA,None,possible,None,None,None


In [ ]:
# CELL 19: Prepare MongoDB Documents

from datetime import datetime, timezone
import json
import pandas as pd
import numpy as np

def clean_value(value):
    """Convert Pandas/NumPy values into MongoDB-compatible Python values."""

    if value is None:
        return None

    if isinstance(value, (dict, list, bool)):
        return value

    if pd.isna(value):
        return None

    if isinstance(value, np.generic):
        return value.item()

    return value


# Check that the required DataFrame exists
if "enriched_df" not in globals():
    raise ValueError("enriched_df not found. Run Cell 18 first.")

# Build MongoDB documents
mongo_docs = []

for _, row in enriched_df.iterrows():

    nutrition = clean_value(row.get("nutrition"))

    if not isinstance(nutrition, dict):
        nutrition = {}

    doc = {
        "productId": clean_value(row.get("productId")),
        "name": clean_value(row.get("name")),
        "brand": clean_value(row.get("brand")),
        "category": clean_value(row.get("category")),
        "subCategory": clean_value(row.get("subCategory")),

        "servingSizeG": clean_value(row.get("servingSizeG")),
        "nutrition": nutrition,

        "ingredientsText": clean_value(row.get("ingredientsText")),

        # Rule-based palm oil detection
        "palmOil": clean_value(row.get("palmOil")),
        "palmOilStatus": clean_value(row.get("palmOilStatus")),

        # Gemini ingredient analysis
        "gemini": {
            "palmOil": clean_value(row.get("geminiPalmOil")),
            "palmOilStatus": clean_value(row.get("geminiPalmOilStatus")),
            "vegetarian": clean_value(row.get("vegetarian")),
            "allergens": clean_value(row.get("allergenInfo")),
            "ingredientSummary": clean_value(row.get("ingredientSummary")),
            "confidence": clean_value(row.get("geminiConfidence"))
        },

        "vitamin": clean_value(row.get("Vitamin")),
        "priceINR": clean_value(row.get("priceINR")),

        "source": ["packaged_foods_india_csv"],

        "dataQuality": clean_value(row.get("dataQuality")),

        "createdAt": datetime.now(timezone.utc)
    }

    # Do not upload records without a product ID or name
    if not doc["productId"] or not doc["name"]:
        continue

    mongo_docs.append(doc)


# Preview
print("=" * 50)
print("MONGODB DOCUMENT PREPARATION")
print("=" * 50)

print("Total valid documents:", len(mongo_docs))
print("Gemini analyzed:", sum(
    1 for doc in mongo_docs
    if doc["gemini"]["ingredientSummary"] is not None
))

if mongo_docs:
    print("\nSample MongoDB document:")
    print(json.dumps(mongo_docs[0], indent=2, default=str))
else:
    raise ValueError("No valid MongoDB documents were generated.")

MONGODB DOCUMENT PREPARATION
Total valid documents: 843
Gemini analyzed: 0

Sample MongoDB document:
{
  "productId": "e013f3bd19cb30a368077757",
  "name": "HIDE & SEEK\u00ae",
  "brand": "PARLE",
  "category": "BISCUIT",
  "subCategory": "COOKIES",
  "servingSizeG": 100.0,
  "nutrition": {
    "caloriesKcal": 479.0,
    "carbs": 73.4,
    "protein": 5.9,
    "totalFat": 18.0,
    "satFat": 9.4,
    "transFat": 0.0,
    "sugars": 32.5,
    "sodiumMg": 114.0,
    "fibre": 0.0,
    "cholesterolMg": 0.0,
    "calciumMg": 0.0,
    "ironMg": 0.0
  },
  "ingredientsText": "REFINED WHEAT FLOUR (MAIDA), CHOCOLATE (23%)  SUGAR, COCOA SOLIDS, COCOA BUTTER, DEXTROSE, EMULSIFIER OF VEGETABLE ORIGIN (SOY LECITHIN) , ARTIFICIAL FLAVOURING SUBSTANCES - VANILLA CAKE, SUGAR, REFINED PALM OIL, INVERT SUGAR SYRUP, RAISING AGENTS 503 (ii ),RAISING AGENTS 500 (ii), IODISED SALT , EMULSIFIER OF VEGETABLE ORIGIN [ 472e]",
  "palmOil": true,
  "palmOilStatus": "confirmed",
  "gemini": {
    "palmOil": null,
 

In [ ]:
from getpass import getpass
from urllib.parse import quote_plus
from pymongo import MongoClient

username = "food_admin"
password = getpass("Enter your actual database password: ")

# Encode credentials safely
username = quote_plus(username)
password = quote_plus(password)

# Use the exact hostname shown in Atlas → Connect → Drivers
cluster = "cluster1.lhnu949.mongodb.net"

MONGODB_URI = (
    f"mongodb+srv://{username}:{password}@{cluster}/"
    "?retryWrites=true&w=majority"
)

print("URI constructed. Testing connection...")

client = MongoClient(
    MONGODB_URI,
    serverSelectionTimeoutMS=15000
)

client.admin.command("ping")

print("MongoDB Atlas connected successfully!")

Enter your actual database password: ··········
URI constructed. Testing connection...


OperationFailure: bad auth : authentication failed, full error: {'ok': 0, 'errmsg': 'bad auth : authentication failed', 'code': 8000, 'codeName': 'AtlasError'}